> **Author:** Kristin Predeck  
> **Date:** April 2026  
> **Purpose:** Ablation study to determine whether near-perfect classification performance reflects true separability or residual leakage from engineered features.  
> **Architecture:** BaselineMLP (input→128→128→1, ReLU, dropout 0.3); same across all conditions for fair comparison.  
> **Reproducibility:** GroupShuffleSplit with random_state=42 ensures identical splits across conditions. 10% training subsample used for computational efficiency; full test set (443K particles) used for evaluation.

# Neural Network Ablation Study

Test the impact of each feature engineering decision by training the same MLP architecture under four conditions:

| Condition | Features | Count | Question |
|---|---|---|---|
| A: Full | CLR + presence + 2 engineered | 56 | Current baseline |
| B: CLR only | CLR-transformed elements | 27 | Does the NN need presence flags and engineered features? |
| C: Raw + presence | Raw elements + presence flags | 54 | Does CLR actually help vs raw compositions? |
| D: Raw only | Raw 27 elements | 27 | Simplest possible — can the NN learn everything from scratch? |

All conditions use the same architecture (56→128→128→1 adjusted for input dim), same split, same scaler, same training procedure. This allows us to test whether feature engineering caused any leakage.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


# Load Data & Build Feature Sets

In [3]:
#load NN-engineered dataset (has CLR, presence, engineered features)
df_nn = pd.read_parquet('../../data/processed/engineered_features_nn.parquet')

#load preprocessed dataset (has raw 27 elements)
df_raw = pd.read_parquet('../../data/processed/preprocessed.parquet')

print(f'NN engineered shape: {df_nn.shape}')
print(f'Raw preprocessed shape: {df_raw.shape}')

NN engineered shape: (2294985, 62)
Raw preprocessed shape: (2294985, 34)


In [4]:
meta_cols = ['stub_id', 'particle_id', 'label', 'target', 'final_class',
             'relevance_class', 'merged_relevance_class']

#raw element cols 
raw_element_cols = [c for c in df_raw.columns if c not in meta_cols]
print(f'Raw element cols ({len(raw_element_cols)}): {raw_element_cols[:5]}...')

#CLR cols
clr_cols = [c for c in df_nn.columns if c.startswith('clr_')]
print(f'CLR cols ({len(clr_cols)}): {clr_cols[:5]}...')

#presence columns
presence_cols = [c for c in df_nn.columns if c.startswith('has_')]
print(f'Presence cols ({len(presence_cols)}): {presence_cols[:5]}...')

#engineered cols (excluding gsr_count)
eng_cols = ['pb_sb_over_non_ba_o_mass', 'log_pb_plus_sb']
print(f'Engineered cols ({len(eng_cols)}): {eng_cols}')

Raw element cols (27): ['o', 's', 'cu', 'ba', 'al']...
CLR cols (27): ['clr_o', 'clr_s', 'clr_cu', 'clr_ba', 'clr_al']...
Presence cols (27): ['has_o', 'has_s', 'has_cu', 'has_ba', 'has_al']...
Engineered cols (2): ['pb_sb_over_non_ba_o_mass', 'log_pb_plus_sb']


In [5]:
conditions = {
    'A: Full': {
        'source': 'nn',
        'cols': clr_cols + presence_cols + eng_cols
    },
    'B: CLR only': {
        'source': 'nn',
        'cols': clr_cols
    },
    'C: Raw + presence': {
        'source': 'raw',
        'cols': raw_element_cols,  # will add presence from nn
        'extra_from_nn': presence_cols
    },
    'D: Raw only': {
        'source': 'raw',
        'cols': raw_element_cols
    }
}

for name, cfg in conditions.items():
    total = len(cfg['cols']) + len(cfg.get('extra_from_nn', []))
    print(f'{name}: {total} features')

A: Full: 56 features
B: CLR only: 27 features
C: Raw + presence: 54 features
D: Raw only: 27 features


**Design rationale:** These four conditions progressively strip away feature engineering layers to isolate where the classification signal originates. If condition D (raw only) matches condition A (full), then all feature engineering is redundant and the NN learns the decision boundary from raw SEM/EDS measurements alone. If there is a gap, the specific feature group responsible can be identified by comparing adjacent conditions.

# Shared Train/Val/Test Split with subsampling

Same split as baseline v2: 60/20/20 with GroupShuffleSplit on stub_id.

In [11]:
y = df_nn['target'].values.astype(np.float32)
groups = df_nn['stub_id'].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(gss1.split(np.zeros(len(y)), y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
groups_tv = groups[trainval_idx]
y_tv = y[trainval_idx]
train_idx_rel, val_idx_rel = next(gss2.split(np.zeros(len(y_tv)), y_tv, groups_tv))
train_idx = trainval_idx[train_idx_rel]
val_idx = trainval_idx[val_idx_rel]

rng = np.random.default_rng(42)
train_idx = rng.choice(train_idx, size=len(train_idx) // 10, replace=False)
val_idx = rng.choice(val_idx, size=len(val_idx) // 10, replace=False)

print(f"Train: {len(train_idx):,} ({y[train_idx].mean():.1%} GSR)")
print(f"Val:   {len(val_idx):,} ({y[val_idx].mean():.1%} GSR)")
print(f"Test:  {len(test_idx):,} ({y[test_idx].mean():.1%} GSR)")

Train: 144,414 (49.6% GSR)
Val:   40,761 (37.4% GSR)
Test:  443,224 (47.4% GSR)


#  Model & Training Infrastructure

In [12]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x)


def build_feature_matrix(condition_cfg, df_nn, df_raw, indices):
    """Extract feature matrix for condition"""
    source = condition_cfg['source']
    cols = condition_cfg['cols']
    
    if source == 'nn':
        X = df_nn[cols].values[indices].astype(np.float32)
    else:  
        X = df_raw[cols].values[indices].astype(np.float32)
    
    extra = condition_cfg.get('extra_from_nn', [])
    if extra:
        X_extra = df_nn[extra].values[indices].astype(np.float32)
        X = np.hstack([X, X_extra])
    
    return X


def train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test,
                        input_dim, max_epochs=20, patience=5, lr=1e-3):
    """Train MLP"""
    model = BaselineMLP(input_dim=input_dim).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                       torch.tensor(y_train, dtype=torch.float32)),
        batch_size=4096, shuffle=True)
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                       torch.tensor(y_val, dtype=torch.float32)),
        batch_size=4096, shuffle=False)
    
    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(max_epochs):
        model.train()
        epoch_loss, n_b = 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb).squeeze(), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_b += 1
        train_losses.append(epoch_loss / n_b)
        
        model.eval()
        vl, nv = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl += criterion(model(xb).squeeze(), yb).item()
                nv += 1
        val_losses.append(vl / nv)
        
        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    
    model.load_state_dict(best_state)
    stopped_epoch = len(train_losses)
    
    model.eval()
    with torch.no_grad():
        test_logits = model(
            torch.tensor(X_test, dtype=torch.float32).to(device)
        ).squeeze().cpu().numpy()
    probs = 1 / (1 + np.exp(-test_logits))
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)
    
    return {
        'roc_auc': roc_auc_score(y_test, probs),
        'pr_auc': average_precision_score(y_test, probs),
        'accuracy': (preds == y_test).mean(),
        'fpr': cm[0,1] / cm[0].sum() if cm[0].sum() > 0 else 0,
        'fnr': cm[1,0] / cm[1].sum() if cm[1].sum() > 0 else 0,
        'fp': cm[0,1],
        'fn': cm[1,0],
        'best_val_loss': best_val_loss,
        'stopped_epoch': stopped_epoch,
        'train_losses': train_losses,
        'val_losses': val_losses
    }

**Architecture choices (held constant across all conditions):**
- 2 hidden layers × 128 units: small enough to train quickly, large enough to learn non-linear boundaries
- Dropout 0.3: moderate regularization to prevent memorization on the 10% subsample
- BCEWithLogitsLoss: numerically stable (combines sigmoid + binary cross-entropy in one operation)
- Adam lr=1e-3: standard default, not tuned (this is a comparison study, not optimization)
- Early stopping patience=3: prevents overfitting while allowing convergence
- Batch size 8192: larger batches for faster iteration on CPU

# Ablation

In [13]:
results = {}

for name, cfg in conditions.items():
    print(f'Training: {name}')
    
    X_tr = build_feature_matrix(cfg, df_nn, df_raw, train_idx)
    X_vl = build_feature_matrix(cfg, df_nn, df_raw, val_idx)
    X_te = build_feature_matrix(cfg, df_nn, df_raw, test_idx)
    
    print(f'Feature dim: {X_tr.shape[1]}')
    
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_vl = sc.transform(X_vl)
    X_te = sc.transform(X_te)
    
    metrics = train_and_evaluate(
        X_tr, y[train_idx], X_vl, y[val_idx], X_te, y[test_idx],
        input_dim=X_tr.shape[1]
    )
    results[name] = metrics
    
    print(f'Stopped at epoch {metrics["stopped_epoch"]}')
    print(f'ROC-AUC:{metrics["roc_auc"]:.4f}  PR-AUC: {metrics["pr_auc"]:.4f}')
    print(f'Accuracy: {metrics["accuracy"]:.4f}  FPR: {metrics["fpr"]:.4f}  FNR: {metrics["fnr"]:.4f}')
    print(f'FP: {metrics["fp"]:,}  FN: {metrics["fn"]:,}')

Training: A: Full
Feature dim: 56
Stopped at epoch 20
ROC-AUC:1.0000  PR-AUC: 1.0000
Accuracy: 0.9979  FPR: 0.0037  FNR: 0.0004
FP: 871  FN: 74
Training: B: CLR only
Feature dim: 27
Stopped at epoch 20
ROC-AUC:0.9999  PR-AUC: 0.9999
Accuracy: 0.9974  FPR: 0.0048  FNR: 0.0000
FP: 1,129  FN: 9
Training: C: Raw + presence
Feature dim: 54
Stopped at epoch 20
ROC-AUC:0.9999  PR-AUC: 0.9999
Accuracy: 0.9974  FPR: 0.0048  FNR: 0.0001
FP: 1,109  FN: 31
Training: D: Raw only
Feature dim: 27
Stopped at epoch 20
ROC-AUC:0.9999  PR-AUC: 0.9998
Accuracy: 0.9961  FPR: 0.0020  FNR: 0.0060
FP: 462  FN: 1,262


# Comparison Table

In [14]:
comparison = []
for name, m in results.items():
    n_feats = len(conditions[name]['cols']) + len(conditions[name].get('extra_from_nn', []))
    comparison.append({
        'Condition': name,
        'Features': n_feats,
        'ROC-AUC': f'{m["roc_auc"]:.4f}',
        'PR-AUC': f'{m["pr_auc"]:.4f}',
        'Accuracy': f'{m["accuracy"]:.4f}',
        'FPR': f'{m["fpr"]:.4f}',
        'FNR': f'{m["fnr"]:.4f}',
        'FP': f'{m["fp"]:,}',
        'FN': f'{m["fn"]:,}',
        'Epochs': m['stopped_epoch']
    })

comp_df = pd.DataFrame(comparison)
print('Ablation Study Results')
print(comp_df.to_string(index=False))

Ablation Study Results
        Condition  Features ROC-AUC PR-AUC Accuracy    FPR    FNR    FP    FN  Epochs
          A: Full        56  1.0000 1.0000   0.9979 0.0037 0.0004   871    74      20
      B: CLR only        27  0.9999 0.9999   0.9974 0.0048 0.0000 1,129     9      20
C: Raw + presence        54  0.9999 0.9999   0.9974 0.0048 0.0001 1,109    31      20
      D: Raw only        27  0.9999 0.9998   0.9961 0.0020 0.0060   462 1,262      20


# Loss Curves Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, m) in enumerate(results.items()):
    ax = axes[i]
    epochs = range(1, len(m['train_losses'])+1)
    ax.plot(epochs, m['train_losses'], label='Train', marker='o', markersize=3)
    ax.plot(epochs, m['val_losses'], label='Val', marker='s', markersize=3)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training vs Validation Loss by Ablation Condition', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("outputs/nn_ablation_study__training_vs_validation_loss_by_ablation_condition.png", dpi=150, bbox_inches="tight")
plt.show()